In [1]:
import psycopg2
import psycopg2.extras
import re
import os
import json
from graphviz import Digraph


# ============================================================
# CONFIGURACIÓN POSTGRESQL
# ============================================================

POSTGRES_CONFIG = {
    "host": "host.docker.internal",
    "port": 5432,
    "user": "admin",
    "password": "12345678",
    "dbname": "spark"
}


# ============================================================
# OPERACIONES DEL PLAN DE EJECUCIÓN POSTGRESQL
# ============================================================

OPERATIONS = {
    "Limit": ("LIMIT", "Limita la cantidad de filas devueltas."),
    "Seq Scan": ("TABLE SCAN", "Lee secuencialmente los registros de una tabla."),
    "Parallel Seq Scan": ("PARALLEL TABLE SCAN", "Lee una tabla utilizando varios procesos."),
    "Index Scan": ("INDEX SCAN", "Recorre un índice y obtiene registros de la tabla."),
    "Index Only Scan": ("INDEX ONLY SCAN", "Obtiene los datos directamente del índice."),
    "Bitmap Index Scan": ("BITMAP INDEX SCAN", "Construye un mapa de posiciones mediante un índice."),
    "Bitmap Heap Scan": ("BITMAP HEAP SCAN", "Obtiene registros utilizando un mapa de posiciones."),
    "Tid Scan": ("TID SCAN", "Accede a registros mediante identificadores físicos."),
    "Nested Loop": ("NESTED LOOP JOIN", "Compara registros mediante búsquedas repetidas."),
    "Hash Join": ("HASH JOIN", "Combina registros utilizando una estructura hash."),
    "Merge Join": ("MERGE JOIN", "Combina entradas previamente ordenadas."),
    "Hash": ("HASH", "Construye una tabla hash para realizar una unión."),
    "Sort": ("SORT", "Ordena los registros."),
    "Incremental Sort": ("INCREMENTAL SORT", "Ordena parcialmente aprovechando un orden existente."),
    "Aggregate": ("AGGREGATE", "Calcula funciones de agregación."),
    "Group": ("GROUP AGGREGATE", "Agrupa registros y calcula agregaciones."),
    "WindowAgg": ("WINDOW AGGREGATE", "Calcula funciones ventana."),
    "Unique": ("REMOVE DUPLICATES", "Elimina registros duplicados."),
    "Materialize": ("MATERIALIZE", "Almacena temporalmente resultados."),
    "Gather": ("GATHER", "Recopila resultados de procesos paralelos."),
    "Gather Merge": ("GATHER MERGE", "Recopila y combina resultados ordenados."),
    "Append": ("APPEND", "Combina resultados de varias entradas."),
    "Merge Append": ("MERGE APPEND", "Combina entradas ordenadas."),
    "Subquery Scan": ("SUBQUERY SCAN", "Lee los resultados de una subconsulta."),
    "CTE Scan": ("CTE SCAN", "Lee los resultados de una expresión CTE."),
    "Function Scan": ("FUNCTION SCAN", "Obtiene registros de una función."),
    "Values Scan": ("VALUES SCAN", "Lee registros definidos mediante VALUES."),
    "Result": ("RESULT", "Genera un resultado calculado."),
    "Memoize": ("MEMOIZE", "Almacena en caché resultados de búsquedas."),
    "SetOp": ("SET OPERATION", "Realiza operaciones sobre conjuntos."),
    "LockRows": ("LOCK ROWS", "Bloquea registros seleccionados."),
    "Recursive Union": ("RECURSIVE UNION", "Combina resultados de una consulta recursiva."),
    "BitmapAnd": ("BITMAP AND", "Combina mapas de índices mediante AND."),
    "BitmapOr": ("BITMAP OR", "Combina mapas de índices mediante OR.")
}


def get_operation(name):
    for key in sorted(OPERATIONS, key=len, reverse=True):
        if name.lower().startswith(key.lower()):
            return OPERATIONS[key]

    return (
        "OPERACIÓN POSTGRESQL",
        "Operación realizada por el optimizador de PostgreSQL."
    )


# ============================================================
# FUNCIONES DE AGREGACIÓN
# ============================================================

AGGREGATE_FUNCTIONS = {
    "COUNT": "Cuenta filas o valores no nulos.",
    "SUM": "Calcula la suma de los valores.",
    "AVG": "Calcula el promedio.",
    "MIN": "Obtiene el valor mínimo.",
    "MAX": "Obtiene el valor máximo.",
    "STRING_AGG": "Concatena valores de un grupo.",
    "ARRAY_AGG": "Construye un arreglo con los valores.",
    "BOOL_AND": "Comprueba si todos los valores son verdaderos.",
    "BOOL_OR": "Comprueba si algún valor es verdadero.",
    "JSON_AGG": "Construye un arreglo JSON.",
    "JSONB_AGG": "Construye un arreglo JSONB.",
    "JSON_OBJECT_AGG": "Construye un objeto JSON.",
    "JSONB_OBJECT_AGG": "Construye un objeto JSONB.",
    "VARIANCE": "Calcula la varianza.",
    "VAR_POP": "Calcula la varianza poblacional.",
    "VAR_SAMP": "Calcula la varianza muestral.",
    "STDDEV": "Calcula la desviación estándar.",
    "STDDEV_POP": "Calcula la desviación estándar poblacional.",
    "STDDEV_SAMP": "Calcula la desviación estándar muestral."
}


# ============================================================
# FUNCIONES VENTANA
# ============================================================

WINDOW_FUNCTIONS = {
    "ROW_NUMBER": "Asigna un número consecutivo a cada fila.",
    "RANK": "Asigna posiciones y deja saltos en los empates.",
    "DENSE_RANK": "Asigna posiciones sin dejar saltos.",
    "NTILE": "Distribuye las filas en grupos.",
    "LAG": "Obtiene un valor de una fila anterior.",
    "LEAD": "Obtiene un valor de una fila posterior.",
    "FIRST_VALUE": "Obtiene el primer valor de la ventana.",
    "LAST_VALUE": "Obtiene el último valor de la ventana.",
    "NTH_VALUE": "Obtiene un valor de una posición específica.",
    "CUME_DIST": "Calcula la distribución acumulada.",
    "PERCENT_RANK": "Calcula el rango porcentual."
}


# ============================================================
# DIVIDIR EXPRESIONES SQL RESPETANDO PARÉNTESIS
# ============================================================

def split_sql_expressions(expression):
    if not expression:
        return []

    parts = []
    current = []
    depth = 0
    quote = None

    for char in expression:
        if quote:
            current.append(char)
            if char == quote:
                quote = None
            continue

        if char in ("'", '"'):
            quote = char
            current.append(char)

        elif char == "(":
            depth += 1
            current.append(char)

        elif char == ")":
            depth -= 1
            current.append(char)

        elif char == "," and depth == 0:
            parts.append("".join(current).strip())
            current = []

        else:
            current.append(char)

    if current:
        parts.append("".join(current).strip())

    return parts


# ============================================================
# RESOLVER NOMBRES FÍSICOS DE TABLAS
# ============================================================

def resolve_table_name(reference, tables):
    reference = reference.replace('"', "").replace("`", "").strip()

    if "." not in reference:
        return reference

    qualifier, column = reference.rsplit(".", 1)

    for item in tables:
        if qualifier.lower() in (
            item["alias"].lower(),
            item["table"].lower(),
            item["table"].split(".")[-1].lower()
        ):
            return item["table"] + "." + column

    return reference


def resolve_column_reference(reference, tables):
    return resolve_table_name(reference, tables)


# ============================================================
# IDENTIFICAR COLUMNAS EN EXPRESIONES
# ============================================================

def extract_columns(expression, tables):
    if not expression:
        return []

    pattern = re.compile(
        r'(?:(?:"?[\w]+"?)\s*\.\s*)?"?[\w]+"?'
    )

    keywords = {
        "COUNT", "SUM", "AVG", "MIN", "MAX",
        "STRING_AGG", "ARRAY_AGG", "DISTINCT",
        "AS", "OVER", "PARTITION", "BY",
        "ORDER", "ROWS", "RANGE", "BETWEEN",
        "PRECEDING", "FOLLOWING", "CURRENT",
        "ROW", "UNBOUNDED", "ASC", "DESC",
        "CASE", "WHEN", "THEN", "ELSE", "END",
        "AND", "OR", "NOT", "IS", "NULL",
        "FILTER", "WHERE", "WITHIN", "GROUP",
        "TRUE", "FALSE", "INTERVAL"
    }

    results = []

    for match in pattern.finditer(expression):
        reference = match.group(0).replace('"', "")

        if reference.upper() in keywords:
            continue

        end = match.end()

        if end < len(expression):
            if expression[end:].lstrip().startswith("("):
                continue

        resolved = resolve_column_reference(
            reference,
            tables
        )

        if resolved not in results:
            results.append(resolved)

    return results


# ============================================================
# ANALIZAR FUNCIONES AGREGADAS Y VENTANA
# ============================================================

def analyze_functions(sql):
    aggregates = []
    windows = []

    window_pattern = re.compile(
        r'\b(' + "|".join(WINDOW_FUNCTIONS) +
        r')\s*\((.*?)\)\s*OVER\s*\((.*?)\)',
        re.I
    )

    for match in window_pattern.finditer(sql):
        function = match.group(1).upper()
        arguments = match.group(2).strip()
        over = match.group(3).strip()

        partition = re.search(
            r'\bPARTITION\s+BY\s+(.*?)(?=\bORDER\s+BY\b|\bROWS\b|\bRANGE\b|$)',
            over,
            re.I
        )

        order = re.search(
            r'\bORDER\s+BY\s+(.*?)(?=\bROWS\b|\bRANGE\b|$)',
            over,
            re.I
        )

        frame = re.search(
            r'\b(ROWS|RANGE|GROUPS)\s+.*$',
            over,
            re.I
        )

        windows.append({
            "function": function,
            "arguments": arguments,
            "over": over,
            "partition_by": partition.group(1).strip() if partition else None,
            "order_by": order.group(1).strip() if order else None,
            "frame": frame.group(0).strip() if frame else None
        })

    aggregate_names = "|".join(AGGREGATE_FUNCTIONS)

    aggregate_pattern = re.compile(
        rf'\b({aggregate_names})\s*\((.*?)\)',
        re.I
    )

    for match in aggregate_pattern.finditer(sql):
        function = match.group(1).upper()

        is_window = bool(
            re.search(
                r'\bOVER\s*\(',
                sql[match.end():match.end() + 300],
                re.I
            )
        )

        aggregates.append({
            "function": function,
            "arguments": match.group(2).strip(),
            "window": is_window
        })

    return aggregates, windows


# ============================================================
# ANALIZAR CONSULTA SQL
# ============================================================

def analyze_sql(sql):
    clean = re.sub(r"\s+", " ", sql).strip()

    tables = []
    joins = []

    reserved = {
        "WHERE", "LEFT", "RIGHT", "INNER",
        "FULL", "OUTER", "CROSS", "JOIN",
        "ON", "ORDER", "GROUP", "LIMIT",
        "HAVING", "OFFSET", "UNION", "WINDOW",
        "FETCH", "FOR", "RETURNING"
    }

    from_match = re.search(
        r'\bFROM\s+((?:"?[\w]+"?\.)?"?[\w]+"?)(?:\s+(?:AS\s+)?("?[\w]+"?))?',
        clean,
        re.I
    )

    if from_match:
        table = from_match.group(1).replace('"', "")
        alias = from_match.group(2)

        if alias:
            alias = alias.replace('"', "")

        if alias and alias.upper() in reserved:
            alias = None

        tables.append({
            "table": table,
            "alias": alias or table.split(".")[-1]
        })

    join_pattern = re.compile(
        r'\b(INNER|LEFT|RIGHT|FULL|CROSS)?\s*'
        r'(?:OUTER\s+)?JOIN\s+'
        r'((?:"?[\w]+"?\.)?"?[\w]+"?)'
        r'(?:\s+(?:AS\s+)?("?[\w]+"?))?'
        r'\s+ON\s+(.*?)'
        r'(?=\s+(?:(?:INNER|LEFT|RIGHT|FULL|CROSS)\s+)?'
        r'(?:OUTER\s+)?JOIN\b|'
        r'\s+WHERE\b|\s+GROUP\s+BY\b|'
        r'\s+HAVING\b|\s+WINDOW\b|'
        r'\s+ORDER\s+BY\b|\s+LIMIT\b|'
        r'\s+OFFSET\b|\s+FETCH\b|$)',
        re.I
    )

    for match in join_pattern.finditer(clean):
        join_type = (match.group(1) or "INNER").upper()
        table = match.group(2).replace('"', "")
        alias = match.group(3)

        if alias:
            alias = alias.replace('"', "")

        if alias and alias.upper() in reserved:
            alias = None

        alias = alias or table.split(".")[-1]

        tables.append({
            "table": table,
            "alias": alias
        })

        joins.append({
            "type": join_type,
            "table": table,
            "alias": alias,
            "condition": match.group(4).strip()
        })

    def clause(pattern):
        match = re.search(pattern, clean, re.I)
        return match.group(1).strip() if match else None

    select = re.search(
        r'\bSELECT\s+(.*?)\s+FROM\b',
        clean,
        re.I
    )

    aggregates, windows = analyze_functions(clean)

    return {
        "tables": tables,
        "joins": joins,
        "select": split_sql_expressions(select.group(1)) if select else [],
        "where": clause(r'\bWHERE\s+(.*?)(?=\s+GROUP\s+BY\b|\s+HAVING\b|\s+WINDOW\b|\s+ORDER\s+BY\b|\s+LIMIT\b|\s+OFFSET\b|\s+FETCH\b|$)'),
        "group_by": clause(r'\bGROUP\s+BY\s+(.*?)(?=\s+HAVING\b|\s+WINDOW\b|\s+ORDER\s+BY\b|\s+LIMIT\b|\s+OFFSET\b|\s+FETCH\b|$)'),
        "having": clause(r'\bHAVING\s+(.*?)(?=\s+WINDOW\b|\s+ORDER\s+BY\b|\s+LIMIT\b|\s+OFFSET\b|\s+FETCH\b|$)'),
        "order_by": clause(r'\bORDER\s+BY\s+(.*?)(?=\s+LIMIT\b|\s+OFFSET\b|\s+FETCH\b|$)'),
        "limit": clause(r'\bLIMIT\s+(\d+|ALL)'),
        "offset": clause(r'\bOFFSET\s+(\d+)'),
        "aggregates": aggregates,
        "windows": windows
    }


# ============================================================
# PARTE 2: INTERPRETACIÓN DEL PLAN POSTGRESQL
# ============================================================

def build_tree(plan_data):
    if isinstance(plan_data, list):
        if not plan_data:
            return None
        plan_data = plan_data[0]

    if "Plan" in plan_data:
        plan_data = plan_data["Plan"]

    def convert(node, depth=0):
        node_type = node.get("Node Type", "Unknown")
        join_type = node.get("Join Type")

        name = node_type + (" " + join_type if join_type else "")

        result = {
            "name": name,
            "node_type": node_type,
            "join_type": join_type,
            "depth": depth,
            "metrics": {
                "cost": node.get("Total Cost"),
                "estimated": node.get("Plan Rows"),
                "start": node.get("Actual Startup Time"),
                "end": node.get("Actual Total Time"),
                "actual": node.get("Actual Rows"),
                "loops": node.get("Actual Loops", 1),
                "width": node.get("Plan Width"),
                "buffers_shared_hit": node.get("Shared Hit Blocks"),
                "buffers_shared_read": node.get("Shared Read Blocks"),
                "buffers_temp_read": node.get("Temp Read Blocks"),
                "buffers_temp_written": node.get("Temp Written Blocks")
            },
            "relation": node.get("Relation Name"),
            "schema": node.get("Schema"),
            "alias": node.get("Alias"),
            "index": node.get("Index Name"),
            "index_cond": node.get("Index Cond"),
            "filter": node.get("Filter"),
            "recheck_cond": node.get("Recheck Cond"),
            "join_filter": node.get("Join Filter"),
            "hash_cond": node.get("Hash Cond"),
            "merge_cond": node.get("Merge Cond"),
            "group_key": node.get("Group Key"),
            "sort_key": node.get("Sort Key"),
            "output": node.get("Output"),
            "subplan_name": node.get("Subplan Name"),
            "workers_planned": node.get("Workers Planned"),
            "workers_launched": node.get("Workers Launched"),
            "children": []
        }

        for child in node.get("Plans", []):
            result["children"].append(
                convert(child, depth + 1)
            )

        return result

    return convert(plan_data)


# ============================================================
# ORDEN DE EJECUCIÓN
# ============================================================

def get_execution_sequence(node, sequence=None):
    if sequence is None:
        sequence = []

    for child in node["children"]:
        get_execution_sequence(child, sequence)

    sequence.append(node)
    return sequence


# ============================================================
# RESOLVER NOMBRE FÍSICO DE TABLA
# ============================================================

def get_physical_table(node, sql_info):
    relation = node.get("relation")

    if not relation:
        return None

    schema = node.get("schema")

    for table in sql_info["tables"]:
        if table["table"].split(".")[-1].lower() == relation.lower():
            if "." not in table["table"] and schema:
                return schema + "." + relation
            return table["table"]

    return (schema + "." + relation) if schema else relation


# ============================================================
# RECOPILAR TABLAS DE UN NODO
# ============================================================

def collect_tables(node, sql_info):
    found = []

    table = get_physical_table(node, sql_info)

    if table:
        found.append(table)

    for child in node["children"]:
        found.extend(collect_tables(child, sql_info))

    return list(dict.fromkeys(found))


# ============================================================
# REEMPLAZAR ALIAS POR NOMBRES FÍSICOS
# ============================================================

def normalize_join_condition(condition, tables):
    if not condition:
        return "Condición no identificada"

    result = condition

    for item in sorted(
        tables,
        key=lambda x: len(x["alias"]),
        reverse=True
    ):
        alias = re.escape(item["alias"])

        result = re.sub(
            r'(?<![\w.])' + alias + r'(?=\s*\.)',
            item["table"],
            result,
            flags=re.I
        )

    return result


# ============================================================
# IDENTIFICAR COLUMNAS DE LAS CONDICIONES
# ============================================================

def extract_join_relationship(condition, tables):
    relationships = []

    if not condition:
        return relationships

    pattern = re.compile(
        r'((?:"?[\w]+"?\.)?"?[\w]+"?)\s*'
        r'(=|<>|!=|>=|<=|>|<)\s*'
        r'((?:"?[\w]+"?\.)?"?[\w]+"?)',
        re.I
    )

    for match in pattern.finditer(condition):
        left = normalize_join_condition(match.group(1), tables)
        right = normalize_join_condition(match.group(3), tables)

        relationships.append({
            "left": left,
            "operator": match.group(2),
            "right": right
        })

    return relationships


# ============================================================
# BUSCAR EL JOIN SQL CORRESPONDIENTE
# ============================================================

def match_physical_join(left_tables, right_tables, sql_info):
    candidates = []

    for join in sql_info["joins"]:
        relationships = extract_join_relationship(
            join["condition"],
            sql_info["tables"]
        )

        score = 0

        for relation in relationships:
            left_table = relation["left"].rsplit(".", 1)[0]
            right_table = relation["right"].rsplit(".", 1)[0]

            if (
                left_table in left_tables
                and right_table in right_tables
            ) or (
                right_table in left_tables
                and left_table in right_tables
            ):
                score += 1

        if score:
            candidates.append({
                "score": score,
                "join": join
            })

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    if (
        len(candidates) > 1
        and candidates[0]["score"] == candidates[1]["score"]
    ):
        return None

    return candidates[0]["join"]


# ============================================================
# PARTE 3: GRAPHVIZ ULTRA-DETALLADO (ESTILO IDENTICO AL DE MYSQL)
# ============================================================

def generate_graphviz(root, sql_info):
    graph = Digraph(
        "Execution Plan Detailed",
        format="png"
    )

    graph.attr(
        rankdir="BT",
        ordering="out",
        bgcolor="white",
        nodesep="0.5",
        ranksep="0.8",
        splines="ortho"
    )

    graph.attr(
        "node",
        shape="box",
        style="filled,rounded",
        fontname="Arial",
        fontsize="10",
        color="#333333",
        penwidth="1.5",
        margin="0.15"
    )

    graph.attr(
        "edge",
        color="#555555",
        arrowsize="0.8"
    )

    COLORS = {
        "LIMIT": "#FFCC80",
        "TABLE SCAN": "#90CAF9",
        "INDEX SCAN": "#80DEEA",
        "MERGE JOIN": "#F48FB1",
        "HASH JOIN": "#B39DDB",
        "NESTED LOOP JOIN": "#CE93D8",
        "SORT": "#FFAB91",
        "GROUP AGGREGATE": "#4DB6AC",
        "WINDOW AGGREGATE": "#7986CB",
        "RESULT": "#B2DFDB",
        "FILTER": "#FFF59D"
    }

    counter = [0]

    def add_nodes(node, parent_id=None):
        counter[0] += 1
        node_id = f"node_{counter[0]}"

        operation, description = get_operation(node["name"])
        color = COLORS.get(operation, "#E0E0E0")
        metrics = node["metrics"]

        # Construcción detallada idéntica al formato solicitado
        label = (
            f"⚡ {operation}\n"
            f"Descripción: {description}\n"
            f"Operación física: {node['name']}"
        )

        table = get_physical_table(node, sql_info)
        if table:
            label += f"\nTabla física: {table}"

        if node.get("index"):
            label += f"\nÍndice: {node['index']}"

        # Si es un nodo de Join, detallar entradas izquierda, derecha y condición ON
        if node["join_type"] or "join" in node["node_type"].lower():
            left_tables = collect_tables(node["children"][0], sql_info) if len(node["children"]) >= 1 else []
            right_tables = collect_tables(node["children"][1], sql_info) if len(node["children"]) >= 2 else []

            label += f"\n\nENTRADA IZQUIERDA:\n" + (", ".join(left_tables) or "Resultado intermedio")
            label += f"\nENTRADA DERECHA:\n" + (", ".join(right_tables) or "Resultado intermedio")

            matching = match_physical_join(left_tables, right_tables, sql_info)
            if matching:
                label += f"\nJOIN SQL: {matching['type']} JOIN"
                label += f"\nON: {normalize_join_condition(matching['condition'], sql_info['tables'])}"

        # Detalles adicionales de filtros, sorts y group keys
        if node.get("merge_cond"):
            label += f"\nMerge Cond: {normalize_join_condition(node['merge_cond'], sql_info['tables'])}"
        elif node.get("hash_cond"):
            label += f"\nHash Cond: {normalize_join_condition(node['hash_cond'], sql_info['tables'])}"
        elif node.get("index_cond"):
            label += f"\nIndex Cond: {normalize_join_condition(node['index_cond'], sql_info['tables'])}"

        if node.get("sort_key"):
            keys = ", ".join(node["sort_key"]) if isinstance(node["sort_key"], list) else node["sort_key"]
            label += f"\nSort Keys: {keys}"

        if node.get("group_key"):
            gkeys = ", ".join(node["group_key"]) if isinstance(node["group_key"], list) else node["group_key"]
            label += f"\nGroup Keys: {gkeys}"

        if node.get("filter"):
            flt = normalize_join_condition(node["filter"], sql_info["tables"])
            label += f"\nFilter: {flt}"

        # Métricas de Coste y Filas Reales
        if metrics["cost"] is not None:
            label += f"\nCoste estimado: {metrics['cost']}"

        if metrics["estimated"] is not None:
            label += f"\nFilas estimadas: {metrics['estimated']}"

        if metrics["actual"] is not None:
            total_real = metrics["actual"] * (metrics["loops"] or 1)
            label += f"\nFilas reales: {total_real}"

        if metrics["start"] is not None and metrics["end"] is not None:
            label += f"\nTiempo: {metrics['start']}..{metrics['end']} ms"

        if metrics["loops"] is not None:
            label += f"\nLoops: {metrics['loops']}"

        graph.node(
            node_id,
            label=label,
            fillcolor=color
        )

        if parent_id:
            graph.edge(parent_id, node_id)

        for child in node["children"]:
            add_nodes(child, node_id)

    add_nodes(root)

    output_path = graph.render(
        filename="execution_plan_postgres",
        directory=".",
        cleanup=True,
        view=False
    )

    print("\nGRAPHVIZ GENERADO:")
    print(os.path.abspath(output_path))

    return output_path


# ============================================================
# EJECUCIÓN PRINCIPAL CONEXIÓN POSTGRESQL
# ============================================================

def execute_query(
    query,
    params=None,
    explain_analyse=True,
    show_query=True
):
    connection = None
    cursor = None

    try:
        connection = psycopg2.connect(**POSTGRES_CONFIG)
        cursor = connection.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

        clean_query = re.sub(
            r"^\s*EXPLAIN\s*\(\s*ANALYZE\s*,?\s*.*?\)\s*",
            "",
            query,
            flags=re.I
        ).strip().rstrip(";")

        if explain_analyse:
            final_query = (
                "EXPLAIN (ANALYZE, VERBOSE, BUFFERS, FORMAT JSON) "
                + clean_query
            )
        else:
            final_query = clean_query

        if show_query:
            print("\n" + "=" * 90)
            print("CONSULTA EJECUTADA")
            print("=" * 90)
            print(clean_query)

        cursor.execute(final_query, params or ())
        results = cursor.fetchall()

        if explain_analyse:
            print("\nPLAN DE EJECUCIÓN POSTGRESQL DETALLADO")
            plan_json = results[0]['QUERY PLAN']
            
            sql_info = analyze_sql(clean_query)
            root_node = build_tree(plan_json)

            # Generar imagen detallada de Graphviz idéntica al formato que solicitaste
            generate_graphviz(root_node, sql_info)

        return results

    except psycopg2.Error as e:
        if connection:
            connection.rollback()
        print("\nERROR POSTGRESQL:", e)
        return None

    finally:
        if cursor:
            cursor.close()
        if connection:
            connection.close()


# ============================================================
# EJEMPLO DE USO
# ============================================================

query = """
SELECT
    c.category_id,
    c.name AS category_name,
    COUNT(p.product_id) AS total_products,
    SUM(p.stock) AS total_stock,
    ROUND(AVG(p.price), 2) AS average_price,
    MIN(p.price) AS minimum_price,
    MAX(p.price) AS maximum_price,
    ROW_NUMBER() OVER (ORDER BY COUNT(p.product_id) DESC) AS numero_fila,
    RANK() OVER (ORDER BY AVG(p.price) DESC) AS ranking_precio,
    DENSE_RANK() OVER (ORDER BY SUM(p.stock) DESC) AS ranking_stock,
    LAG(AVG(p.price)) OVER (ORDER BY c.category_id) AS precio_categoria_anterior,
    LEAD(AVG(p.price)) OVER (ORDER BY c.category_id) AS precio_categoria_siguiente,
    SUM(COUNT(p.product_id)) OVER () AS total_general
FROM categories c
INNER JOIN products p ON c.category_id = p.category_id
GROUP BY c.category_id, c.name
HAVING COUNT(p.product_id) >= 1 AND SUM(p.stock) > 0 AND AVG(p.price) > 0
ORDER BY c.category_id, total_products DESC, average_price DESC
LIMIT 100
OFFSET 1;
"""

result = execute_query(
    query=query,
    explain_analyse=True,
    show_query=True
)


CONSULTA EJECUTADA
SELECT
    c.category_id,
    c.name AS category_name,
    COUNT(p.product_id) AS total_products,
    SUM(p.stock) AS total_stock,
    ROUND(AVG(p.price), 2) AS average_price,
    MIN(p.price) AS minimum_price,
    MAX(p.price) AS maximum_price,
    ROW_NUMBER() OVER (ORDER BY COUNT(p.product_id) DESC) AS numero_fila,
    RANK() OVER (ORDER BY AVG(p.price) DESC) AS ranking_precio,
    DENSE_RANK() OVER (ORDER BY SUM(p.stock) DESC) AS ranking_stock,
    LAG(AVG(p.price)) OVER (ORDER BY c.category_id) AS precio_categoria_anterior,
    LEAD(AVG(p.price)) OVER (ORDER BY c.category_id) AS precio_categoria_siguiente,
    SUM(COUNT(p.product_id)) OVER () AS total_general
FROM categories c
INNER JOIN products p ON c.category_id = p.category_id
GROUP BY c.category_id, c.name
HAVING COUNT(p.product_id) >= 1 AND SUM(p.stock) > 0 AND AVG(p.price) > 0
ORDER BY c.category_id, total_products DESC, average_price DESC
LIMIT 100
OFFSET 1

PLAN DE EJECUCIÓN POSTGRESQL DETALLADO

GRAPH

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text


def get_pandas_connection(config):

    connection_string = (
        f"postgresql+psycopg2://"
        f"{config['user']}:"
        f"{config['password']}@"
        f"{config['host']}:"
        f"{config['port']}/"
        f"{config['dbname']}"
    )

    return create_engine(connection_string)


# ============================================================
# EJECUTAR QUERY POSTGRESQL CON PANDAS
# ============================================================

def execute_query_pandas(
    query,
    params=None,
    show_results=False,
    config=None
):

    engine = None

    try:

        engine = get_pandas_connection(config)

        df = pd.read_sql_query(
            text(query),
            engine,
            params=params
        )

        if show_results:

            print("\nRESULTADOS DE PANDAS")
            print("=" * 90)

            print(df.to_string(index=False))

            print("\nDIMENSIONES DEL DATAFRAME")
            print("Filas:", df.shape[0])
            print("Columnas:", df.shape[1])

            print("\nTIPOS DE DATOS")
            print(df.dtypes)

        return df

    except Exception as e:

        print("ERROR PANDAS/POSTGRESQL:", e)

        return None

    finally:

        if engine:
            engine.dispose()


# ============================================================
# EJEMPLO DE USO
# ============================================================
df = execute_query_pandas(
    query=query,
    show_results=False,
    config=POSTGRES_CONFIG
)

df

,category_id,category_name,total_products,total_stock,average_price,minimum_price,maximum_price,numero_fila,ranking_precio,ranking_stock,precio_categoria_anterior,precio_categoria_siguiente,total_general
0,2,Software,20,19980,169.71,19.99,899.00,6,5,1,155.8625,186.6150,330.0
1,3,Furniture,20,602,186.62,45.99,549.00,5,4,15,169.7140,31.0670,330.0
2,4,Accessories,20,2570,31.07,9.99,89.99,13,12,3,186.6150,31.7390,330.0
3,5,Sports,20,2195,31.74,9.99,79.99,12,11,8,31.0670,27.1890,330.0
4,6,Toys & Games,20,2070,27.19,8.99,89.99,15,14,10,31.7390,14.2645,330.0
5,7,Books & Stationery,20,3285,14.26,4.99,29.99,1,18,2,27.1890,45.4615,330.0
6,8,Home & Kitchen,20,1835,45.46,9.99,189.99,9,8,12,14.2645,23.8420,330.0
7,9,Beauty & Personal Care,20,2465,23.84,7.99,65.00,3,16,4,45.4615,27.7910,330.0
8,10,Automotive,20,2365,27.79,9.99,69.99,14,13,5,23.8420,23.5665,330.0
9,11,Pet Supplies,20,2105,23.57,8.99,59.99,4,17,9,27.7910,24.5410,330.0
